In [ ]:
import pandas as pd

In [ ]:
#ファイル読み込み
df = pd.read_csv('../000.data/train/train_A.tsv', sep="\t", parse_dates=["time_stamp"])
tdf = pd.read_csv('../000.data/test.tsv', sep="\t")

In [ ]:
# 末尾が "_A" のテストデータだけを抽出
tdf_A = tdf[tdf["user_id"].str.endswith("_A")]

In [ ]:
# 行動に対応する重みを定義
event_weights = {
    3: 3,  # 購入
    2: 2,  # 広告クリック
    1: 1,  # 詳細ページの閲覧
    0: 0 # カートに入れる
}

In [ ]:
# スコア計算
def calculate_score(row):
    if row["event_type"] == 3 and row["ad"] != 1:
        return 0  # 購入は広告クリック(ad==1)時のみ有効
    return event_weights.get(row["event_type"], 0)

df["score"] = df.apply(calculate_score, axis=1)

In [ ]:
# ユーザーごとの時系列特徴量の追加
df["latest_action_time"] = df.groupby("user_id")["time_stamp"].transform("max")
df["time_since_last_action"] = (df["latest_action_time"] - df["time_stamp"]).dt.total_seconds()
df["hour_of_day"] = df["time_stamp"].dt.hour
df["day_of_week"] = df["time_stamp"].dt.dayofweek

In [ ]:
# 過去7日間の行動回数

# 各ユーザーのデータを時系列順にソート
df_sorted = df.sort_values(["user_id", "time_stamp"])

# ローリングカウントを計算
rolling_counts = (
    df_sorted.set_index("time_stamp")
    .groupby("user_id")["event_type"]
    .rolling("7d")
    .count()
    .reset_index()
    .rename(columns={"event_type": "actions_last_7d"})
)

# `df` にマージ
df = df.merge(rolling_counts, on=["user_id", "time_stamp"], how="left")

In [ ]:
# 夜間（0-6時）の行動割合
df["night_activity"] = (df["hour_of_day"] < 6).astype(int)
df["night_activity_ratio"] = df.groupby("user_id")["night_activity"].transform("mean")

In [ ]:
# 夜間行動割合を加味したスコア
df["adjusted_night_score"] = df["score"] * (1 + df["night_activity_ratio"])

In [ ]:
# 週末（土日）の行動割合
df["weekend_activity"] = (df["day_of_week"] >= 5).astype(int)
df["weekend_activity_ratio"] = df.groupby("user_id")["weekend_activity"].transform("mean")

In [ ]:
# 週末行動割合を加味したスコア
df["adjusted_weekend_score"] = df["score"] * (1 + df["weekend_activity_ratio"])

In [ ]:
# ユーザー×商品ごとの行動履歴
df["user_product_interaction"] = df.groupby(["user_id", "product_id"])["event_type"].transform("count")
df["user_last_view_time"] = df.groupby(["user_id", "product_id"])["time_stamp"].transform("max")
df["user_time_since_last_view"] = (df["latest_action_time"] - df["user_last_view_time"]).dt.total_seconds()

In [ ]:
# ユーザー×商品ごとの行動回数を考慮
df["adjusted_interaction_score"] = df["score"] * (1 + df["user_product_interaction"] / df["user_product_interaction"].max())

In [ ]:
# ユーザーが過去に同じ商品を何回閲覧したか
df["user_product_view_count"] = df.groupby(["user_id", "product_id"])["event_type"].transform("count")

In [ ]:
# ユーザー商品閲覧数を加味したスコア
df["adjusted_viewcount_score"] = df["score"] * (1 + df["user_product_view_count"])

In [ ]:
# 商品に対する一貫した関心度
df["user_interest_level"] = df.groupby(["user_id", "product_id"])["time_stamp"].transform(lambda x: x.diff().mean())
df["user_interest_level"] = pd.to_timedelta(df["user_interest_level"])
df["user_interest_level"] = df["user_interest_level"].dt.total_seconds()  # 秒単位に変換

In [ ]:
# ユーザー関心度を加味したスコア
df["adjusted_interestlevel_score"] = df["score"] * (1 + df["user_interest_level"])

In [ ]:
# 商品ごとの購入率
df["product_purchase_rate"] = product_conversion_rate = df[df["event_type"] == 3].groupby("product_id")["event_type"].count() / df.groupby("product_id")["event_type"].count()

In [ ]:
# 購入率を加味したスコア
df["adjusted_purchaserate_score"] = df["score"] * (1 + df["product_purchase_rate"])

In [ ]:
# 最終スコア（夜間・週末を考慮）
df["final_score"] = (df["adjusted_night_score"] + df["adjusted_weekend_score"]) / 2

In [ ]:
# 最終スコアに加味_1
df["final_score"] = (df["final_score"] + df["adjusted_interaction_score"]) / 2

In [ ]:
# 最終スコアに加味_2
df["final_score"] = (df["final_score"] + df["adjusted_viewcount_score"]) / 2

In [ ]:
# 最終スコアに加味_3
df["final_score"] = (df["final_score"] + df["adjusted_interestlevel_score"]) / 2

In [ ]:
# 最終スコアに加味_4
df["final_score"] = (df["final_score"] + df["adjusted_purchaserate_score"]) / 2

In [ ]:
# ユーザー×商品ごとのスコア集計
user_product_score = df.groupby(["user_id", "product_id"], as_index=False)["final_score"].sum()

In [ ]:
# ユーザーごとにスコア降順、同スコアなら product_id 昇順で順位付け
user_product_score.sort_values(by=["user_id", "final_score", "product_id"], ascending=[True, False, True], inplace=True)
user_product_score["rank"] = user_product_score.groupby("user_id")["final_score"].rank(method="first", ascending=False).astype(int)

In [ ]:
# 必要なカラムだけを残して重複を削除
user_product_score = user_product_score[["user_id", "product_id", "rank"]].drop_duplicates()

In [ ]:
#前処理結果をcsvへ
user_product_score.to_csv('./train/train_A.csv', index=False)
tdf_A.to_csv('./test/test_A.csv', index=False)